# Building Decision Trees

## Goal

Build a decision tree from scratch to understand how tree-based machine learning models learn rules, evaluate splits, and make predictions.

The project starts from the simplest possible model and evolves step by step toward numerical splits, deeper trees, and boosting.

---

## Version 5

Extend the missing-aware decision tree into a simple gradient boosting model.

The model will:

- start from a constant prediction
- compute residual errors
- train shallow trees on the residuals
- update the current predictions with each new tree
- repeat the process for several boosting rounds
- keep native missing-value handling inside each tree
- convert the final scores into binary Titanic predictions
- generate a new Titanic submission

In [1]:
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

In [3]:
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Feature engineering

def add_title_feature(df):
    df = df.copy()

    names = df["Name"].str.lower().str.strip()
    df["Title"] = names.str.extract(r" ([a-z]+)\.", expand=False)

    return df

train = add_title_feature(train)
test = add_title_feature(test)

In [5]:
# Define target and feature types

TARGET = "Survived"

NUMERIC_FEATURES = ["Age", "Fare"]
CATEGORICAL_FEATURES = ["Sex", "Title", "Pclass"]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [6]:
# Check missing values

print("Train missing values:")
print(train[FEATURES].isna().sum())

print("\nTest missing values:")
print(test[FEATURES].isna().sum())

Train missing values:
Age       177
Fare        0
Sex         0
Title       0
Pclass      0
dtype: int64

Test missing values:
Age       86
Fare       1
Sex        0
Title      0
Pclass     0
dtype: int64


## Data preparation

We create the training features, the target, and the test features used by the model.

In [7]:
# Prepare training and test features

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]

In [8]:
X_train.head()

,Age,Fare,Sex,Title,Pclass
PassengerId,,,,,
1,22.0,7.2500,male,mr,3
2,38.0,71.2833,female,mrs,1
3,26.0,7.9250,female,miss,3
4,35.0,53.1000,female,mrs,1
5,35.0,8.0500,male,mr,3


## Building the model

We now extend the missing-aware decision tree into a simple gradient boosting model.

Instead of relying on one tree, the model will build several shallow trees that progressively correct prediction errors.

In [9]:
# Fit a constant predictor

def fit_constant(y):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    return sum(y) / len(y)

### Initial prediction

Gradient boosting starts with the same constant prediction for every training observation.

With squared error, this initial prediction is the average target value.

In [10]:
# Compute the initial constant prediction

initial_prediction = fit_constant(y_train)

initial_prediction

0.3838383838383838

### Residuals

The first tree does not predict the final target directly.

It learns the residual errors left by the current prediction.

In [11]:
# Compute residual errors

residuals = y_train - initial_prediction

residuals.head()

PassengerId
1   -0.383838
2    0.616162
3    0.616162
4    0.616162
5   -0.383838
Name: Survived, dtype: float64

In [12]:
# Compute mean squared error

def mean_squared_error(y, prediction):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    total_error = 0

    for value in y:
        total_error += (value - prediction) ** 2

    return total_error / len(y)

In [13]:
# Split target values using a categorical equality rule

def split_categorical(x, y, category):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value == category:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

In [14]:
# Split target values using a numerical threshold

def split_numeric(x, y, threshold):
    y_left = []
    y_right = []
    y_missing = []

    for feature_value, target_value in zip(x, y):
        if pd.isna(feature_value):
            y_missing.append(target_value)
        elif feature_value <= threshold:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right, y_missing

### Recursive dataset splitting

A decision stump only needs the target values in each branch.

A recursive tree also needs the feature rows that reach each branch, because every branch may search for another split.

In [15]:
# Split the full dataset using a learned rule

def split_dataset(
    X,
    y,
    feature,
    split_type,
    split_value,
    missing_go_left
):
    if split_type == "categorical":
        go_left = X[feature] == split_value

    else:
        if missing_go_left:
            go_left = X[feature].isna() | (X[feature] <= split_value)
        else:
            go_left = X[feature].notna() & (X[feature] <= split_value)

    X_left = X[go_left]
    y_left = y[go_left]

    X_right = X[~go_left]
    y_right = y[~go_left]

    return X_left, y_left, X_right, y_right

### Leaf nodes

A leaf is a terminal node that no longer splits the data.

It stores the prediction learned from the target values that reach it.

In [16]:
# Create a leaf node

def make_leaf(y):
    prediction = fit_constant(y)

    return {
        "type": "leaf",
        "prediction": prediction
    }

### Stopping conditions

A recursive tree cannot split forever.

A branch becomes a leaf when the target is already pure, when the maximum depth is reached, or when no useful split can be found.

In [17]:
# Check whether a branch should stop growing

def should_stop(y, depth, max_depth):
    if depth >= max_depth:
        return True

    if len(set(y)) == 1:
        return True

    return False

In [18]:
# Compute the weighted MSE after a categorical split

def categorical_split_mse(x, y, category):
    y_left, y_right = split_categorical(x, y, category)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [19]:
# Compute the weighted MSE after a numerical split

def numeric_split_mse(x, y, threshold, missing_go_left):
    y_left, y_right, y_missing = split_numeric(x, y, threshold)

    if missing_go_left:
        y_left = y_left + y_missing
    else:
        y_right = y_right + y_missing

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [20]:
# Compute the error reduction produced by a categorical split

def categorical_split_gain(x, y, category):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = categorical_split_mse(x, y, category)

    return base_mse - split_mse

In [21]:
# Compute the error reduction produced by a numerical split

def numeric_split_gain(x, y, threshold, missing_go_left):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = numeric_split_mse(x, y, threshold, missing_go_left)

    return base_mse - split_mse

### Numerical split search

For numerical features, the model must find the threshold that produces the largest gain.

Instead of testing every possible number, we sort the unique non-missing values and test the midpoints between consecutive values.

For each threshold, the model also evaluates whether missing values should go left or right and keeps the direction with the largest gain.

In [22]:
# Find the best numerical split for one feature

def find_best_numeric_split(x, y):
    values = []

    for value in x:
        if not pd.isna(value) and value not in values:
            values.append(value)

    values.sort()

    if len(values) < 2:
        return None, None, float("-inf")

    thresholds = []

    for i in range(len(values) - 1):
        midpoint = (values[i] + values[i + 1]) / 2
        thresholds.append(midpoint)

    best_threshold = None
    best_missing_go_left = None
    best_gain = float("-inf")

    for threshold in thresholds:
        gain_left = numeric_split_gain(x, y, threshold, True)
        gain_right = numeric_split_gain(x, y, threshold, False)

        if gain_left > best_gain:
            best_threshold = threshold
            best_missing_go_left = True
            best_gain = gain_left

        if gain_right > best_gain:
            best_threshold = threshold
            best_missing_go_left = False
            best_gain = gain_right

    return best_threshold, best_missing_go_left, best_gain

In [23]:
# Find the best categorical split for one feature

def find_best_categorical_split(x, y):
    categories = []

    for value in x:
        if value not in categories:
            categories.append(value)

    if len(categories) < 2:
        return None, float("-inf")

    best_category = None
    best_gain = float("-inf")

    for category in categories:
        gain = categorical_split_gain(x, y, category)

        if gain > best_gain:
            best_category = category
            best_gain = gain

    return best_category, best_gain

In [24]:
# Find the best split across all features

def find_best_feature_split(X, y):
    best_feature = None
    best_split_type = None
    best_split_value = None
    best_missing_go_left = None
    best_gain = float("-inf")

    for feature in X:
        if feature in NUMERIC_FEATURES:
            split_value, missing_go_left, gain = find_best_numeric_split(X[feature], y)
            split_type = "numeric"
        else:
            split_value, gain = find_best_categorical_split(X[feature], y)
            split_type = "categorical"
            missing_go_left = None

        if gain > best_gain:
            best_feature = feature
            best_split_type = split_type
            best_split_value = split_value
            best_missing_go_left = missing_go_left
            best_gain = gain

    return (
        best_feature,
        best_split_type,
        best_split_value,
        best_missing_go_left,
        best_gain
    )

In [25]:
# Build a recursive decision tree

def build_tree(X, y, depth=0, max_depth=3):
    if should_stop(y, depth, max_depth):
        return make_leaf(y)

    (
        best_feature,
        best_split_type,
        best_split_value,
        best_missing_go_left,
        best_gain
    ) = find_best_feature_split(X, y)

    if best_gain <= 0:
        return make_leaf(y)

    X_left, y_left, X_right, y_right = split_dataset(
        X,
        y,
        best_feature,
        best_split_type,
        best_split_value,
        best_missing_go_left
    )

    left_child = build_tree(X_left, y_left, depth + 1, max_depth)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth)

    return {
        "type": "node",
        "feature": best_feature,
        "split_type": best_split_type,
        "split_value": best_split_value,
        "missing_go_left": best_missing_go_left,
        "gain": best_gain,
        "left": left_child,
        "right": right_child
    }

### First boosting tree

The first shallow tree is trained on the residuals instead of the original target.

Its goal is to learn how to correct the current predictions.

In [26]:
# Train the first tree on residuals

first_tree = build_tree(X_train, residuals, max_depth=2)

first_tree

{'type': 'node',
 'feature': 'Title',
 'split_type': 'categorical',
 'split_value': 'mr',
 'missing_go_left': None,
 'gain': 0.07133502379453835,
 'left': {'type': 'node',
  'feature': 'Fare',
  'split_type': 'numeric',
  'split_value': 26.26875,
  'missing_go_left': True,
  'gain': 0.009863136432909217,
  'left': {'type': 'leaf', 'prediction': -0.28383838383838383},
  'right': {'type': 'leaf', 'prediction': -0.053129722421060996}},
 'right': {'type': 'node',
  'feature': 'Pclass',
  'split_type': 'categorical',
  'split_value': 3,
  'missing_go_left': None,
  'gain': 0.03947085427797725,
  'left': {'type': 'leaf', 'prediction': 0.09871975569649988},
  'right': {'type': 'leaf', 'prediction': 0.4973497349734973}}}

### Learned tree structure

The trained tree keeps the same missing-aware recursive structure introduced in Version 4.

Each internal node stores a learned rule, while each leaf stores the average residual of the samples that reached it.

Numerical nodes also store `missing_go_left`, which records the learned direction for missing values.

The same feature can appear more than once because every node works on a different subset of the data.

### Recursive prediction

To predict with one tree, an observation starts at the root and follows the learned rules.

At each node, it moves left or right until it reaches a leaf.

In Version 5, the leaf value represents a correction that will be added to the current boosting prediction.

In [27]:
# Predict one observation by traversing the tree

def predict_one(row, tree):
    if tree["type"] == "leaf":
        return tree["prediction"]

    feature_value = row[tree["feature"]]

    if tree["split_type"] == "categorical":
        go_left = feature_value == tree["split_value"]

    else:
        if pd.isna(feature_value):
            go_left = tree["missing_go_left"]
        else:
            go_left = feature_value <= tree["split_value"]

    if go_left:
        return predict_one(row, tree["left"])
    else:
        return predict_one(row, tree["right"])

In [28]:
# Predict tree values for all observations

def predict_tree_values(X, tree):
    predictions = []

    for _, row in X.iterrows():
        predictions.append(predict_one(row, tree))

    return predictions

## Gradient boosting

The first tree has learned a correction to the initial prediction.

We now control how much of that correction is added using a learning rate.

In [29]:
# Update predictions with the first tree

learning_rate = 0.1

first_tree_predictions = pd.Series(
    predict_tree_values(X_train, first_tree),
    index=y_train.index
)

current_predictions = (
    initial_prediction
    + learning_rate * first_tree_predictions
)

current_predictions.head()

PassengerId
1    0.355455
2    0.433573
3    0.393710
4    0.433573
5    0.355455
dtype: float64

In [30]:
# Compute the remaining residuals

residuals = y_train - current_predictions

residuals.head()

PassengerId
1   -0.355455
2    0.566427
3    0.606290
4    0.566427
5   -0.355455
dtype: float64

In [31]:
# Train the second tree on the remaining residuals

second_tree = build_tree(X_train, residuals, max_depth=2)

second_tree

{'type': 'node',
 'feature': 'Sex',
 'split_type': 'categorical',
 'split_value': 'male',
 'missing_go_left': None,
 'gain': 0.05809976011287438,
 'left': {'type': 'node',
  'feature': 'Title',
  'split_type': 'categorical',
  'split_value': 'master',
  'missing_go_left': None,
  'gain': 0.008976457453123199,
  'left': {'type': 'leaf', 'prediction': 0.1693307412136563},
  'right': {'type': 'leaf', 'prediction': -0.20367126098084665}},
 'right': {'type': 'node',
  'feature': 'Pclass',
  'split_type': 'categorical',
  'split_value': 3,
  'missing_go_left': None,
  'gain': 0.041167903573014974,
  'left': {'type': 'leaf', 'prediction': 0.10628964059196622},
  'right': {'type': 'leaf', 'prediction': 0.5134854661936782}}}

In [32]:
# Update predictions with the second tree

second_tree_predictions = pd.Series(
    predict_tree_values(X_train, second_tree),
    index=y_train.index
)

current_predictions = (
    current_predictions
    + learning_rate * second_tree_predictions
)

current_predictions.head()

PassengerId
1    0.335087
2    0.484922
3    0.404339
4    0.484922
5    0.335087
dtype: float64

### Boosting loop

The same process can now be repeated automatically.

At each boosting round, the model computes residuals, trains a shallow tree, and updates the current predictions.

In [33]:
# Train a simple gradient boosting model

def fit_gradient_boosting(
    X,
    y,
    n_trees=5,
    learning_rate=0.1,
    max_depth=2
):
    initial_prediction = fit_constant(y)

    current_predictions = pd.Series(
        initial_prediction,
        index=y.index
    )

    trees = []

    for _ in range(n_trees):
        residuals = y - current_predictions

        tree = build_tree(
            X,
            residuals,
            max_depth=max_depth
        )

        tree_predictions = pd.Series(
            predict_tree_values(X, tree),
            index=y.index
        )

        current_predictions = (
            current_predictions
            + learning_rate * tree_predictions
        )

        trees.append(tree)

    return {
        "initial_prediction": initial_prediction,
        "learning_rate": learning_rate,
        "trees": trees
    }

In [34]:
# Train the Version 5 gradient boosting model

boosting_model = fit_gradient_boosting(
    X_train,
    y_train,
    n_trees=5,
    learning_rate=0.1,
    max_depth=2
)

len(boosting_model["trees"])

5

### Boosting prediction

To predict with the boosting model, we start from the initial constant prediction.

Then we add the correction produced by every trained tree.

In [35]:
# Predict continuous scores with the boosting model

def predict_gradient_boosting_values(X, model):
    predictions = pd.Series(
        model["initial_prediction"],
        index=X.index
    )

    for tree in model["trees"]:
        tree_predictions = pd.Series(
            predict_tree_values(X, tree),
            index=X.index
        )

        predictions = (
            predictions
            + model["learning_rate"] * tree_predictions
        )

    return predictions

In [36]:
# Convert boosting scores into binary classes

def predict_gradient_boosting(X, model, threshold=0.5):
    scores = predict_gradient_boosting_values(X, model)
    predictions = []

    for score in scores:
        if score >= threshold:
            predictions.append(1)
        else:
            predictions.append(0)

    return predictions

In [37]:
# Generate binary predictions for the Kaggle test set

test_predictions = predict_gradient_boosting(
    X_test,
    boosting_model
)

In [38]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": test_predictions
})

submission.to_csv("submission_v5.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Notes

This fifth version extends the missing-aware decision tree into a simple gradient boosting model.

- The model starts from the average target value.
- Residuals represent the errors left by the current predictions.
- Shallow trees are trained sequentially on the residuals.
- Each tree contributes a scaled correction controlled by the learning rate.
- The boosting loop repeats this process for multiple trees.
- Each tree keeps the missing-value handling introduced in Version 4.
- Final continuous scores are converted into binary classes using a threshold of `0.5`.
- The implementation uses squared-error residuals to keep the boosting logic simple and transparent.

This version completes the five-stage Building Decision Trees project.